In [1]:
import os
from os.path import join
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image

train_img_dir = "../input/train/train"
test_img_dir = "../input/test/test"

train_data = pd.read_csv("../input/train.csv")

# filter only image files (skip possible nested directories)
test_ids = sorted([f for f in os.listdir(test_img_dir) if f.lower().endswith(".jpg")])
test_img_paths = [join(test_img_dir, img) for img in test_ids]

img_size = 32


def prep_imgs(img_paths, img_height=img_size, img_width=img_size):
    # Load images with Pillow, resize, convert to array and scale to [0,1]
    arr = []
    for p in img_paths:
        img = Image.open(p).convert("RGB")
        img = img.resize((img_width, img_height))
        arr.append(np.array(img, dtype=np.float32) / 255.0)
    return np.stack(arr)


train_img_paths = [join(train_img_dir, img) for img in train_data["id"]]
X_train = prep_imgs(train_img_paths)
y_train = train_data["has_cactus"].values.astype(np.int32)




2026-05-08 00:16:39.646062: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778199399.659504      55 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778199399.663776      55 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-08 00:16:39.681288: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
model = tf.keras.Sequential(
    [
        tf.keras.layers.Conv2D(
            25,
            kernel_size=2,
            strides=2,
            activation="relu",
            input_shape=(img_size, img_size, 3),
        ),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
        tf.keras.layers.Conv2D(25, kernel_size=2, strides=2, activation="relu"),
        tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=2),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(250, activation="relu"),
        tf.keras.layers.Dense(2, activation="softmax"),
    ]
)

model.compile(
    loss="sparse_categorical_crossentropy", optimizer="adam", metrics=["accuracy"]
)

model.summary()




/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-05-08 00:16:51.595766: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1778199411.596456      55 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:86:00.0, compute capability: 8.6


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 16, 16, 25)     │           325 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 8, 8, 25)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 4, 4, 25)       │         2,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 2, 2, 25)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 250)            │        25,250 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │           502 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 28,602 (111.73 KB)

 Trainable params: 28,602 (111.73 KB)

 Non-trainable params: 0 (0.00 B)

In [3]:
model.fit(X_train, y_train, batch_size=100, epochs=4, validation_split=0.2, verbose=2)




Epoch 1/4


I0000 00:00:1778199412.971817     108 service.cc:148] XLA service 0x7fa890004f50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778199412.971850     108 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-05-08 00:16:52.988815: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778199413.073391     108 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-05-08 00:16:53.670380: I external/local_xla/xla/stream_executor/cuda/cuda_asm_compiler.cc:397] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_250', 120 bytes spill stores, 120 bytes spill loads

E0000 00:00:1778199414.880845     108 buffer_comparator.cc:157] Difference at 0: inf, expected 4.93811
E0000 00:00:1778199414.880880     108 buffer_comparator.cc:157] Difference at 1: inf, expected 4.8369

114/114 - 7s - 60ms/step - accuracy: 0.8099 - loss: 0.3993 - val_accuracy: 0.9252 - val_loss: 0.2033
Epoch 2/4
114/114 - 0s - 2ms/step - accuracy: 0.9253 - loss: 0.1909 - val_accuracy: 0.9266 - val_loss: 0.1764
Epoch 3/4
114/114 - 0s - 2ms/step - accuracy: 0.9344 - loss: 0.1681 - val_accuracy: 0.9383 - val_loss: 0.1526
Epoch 4/4
114/114 - 0s - 2ms/step - accuracy: 0.9419 - loss: 0.1522 - val_accuracy: 0.9506 - val_loss: 0.1342


In [4]:
X_test = prep_imgs(test_img_paths)
preds_proba = model.predict(X_test)[:, 1]  # probability of class "has_cactus"

submission = pd.DataFrame({"id": test_ids, "has_cactus": preds_proba})
submission.to_csv("submission.csv", index=False)

104/104 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step
